# 🐍 Proje #21 — YouTube Video & Ses İndirme Botu

Bu projede, **yt-dlp** motorunu kullanarak YouTube üzerinden videoları veya sesleri yüksek kalitede bilgisayarımıza indiren modern ve donmayan (thread-safe) bir Tkinter masaüstü arayüzü geliştirdik.

### 🎯 Öğrenilecek Konular:
- **yt-dlp** kütüphanesi ile video sorgulama ve indirme
- **threading** ile ağ istekleri yaparken arayüzün donmasını engelleme (asenkron çalışma mantığı)
- **Pillow (PIL)** ve **requests** kullanarak HTTP üzerinden görsel (Thumbnail) çekip Tkinter'da gösterme
- **FFmpeg** entegrasyonu: Video ve ses birleştirme ile MP3 dönüştürme
- Dinamik hata yönetimi: FFmpeg yüklü olmadığında çökme yaşamadan m4a/best formatlarına geri çekilme (fallback)

## 📦 Gerekli Kütüphaneler ve Kurulum

Bu projede yerleşik python kütüphanelerinin (`tkinter`, `os`, `re`, `shutil`, `threading`) yanı sıra bazı harici kütüphanelere ihtiyaç duyarız:

```bash
pip install requests pillow yt-dlp
```

### 🛠️ FFmpeg Kurulumu:
- **Ses dönüştürme (MP3)** ve **1080p gibi yüksek çözünürlüklü videoların birleştirilmesi** için bilgisayarınızda **FFmpeg** kurulu olmalıdır.
- FFmpeg'i indirdikten sonra klasör yolunu (Örn: `C:\ffmpeg\bin`) sisteminizin `PATH` çevre değişkenlerine ekleyebilir ya da programın en başındaki `FFMPEG_LOCATION` değişkenini güncelleyebilirsiniz.

## 🛠️ Modüllerin İçe Aktarılması ve Yardımcı Fonksiyonlar

Uygulamamızın çalışması için gerekli modülleri import edelim. Ayrıca dosya adlarındaki geçersiz karakterleri temizleyecek `sanitize_filename` ve FFmpeg yolunu bulacak `get_ffmpeg_path` fonksiyonlarını tanımlayalım.

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import os
import re
import io
import shutil
import threading
import requests
from PIL import Image, ImageTk
from yt_dlp import YoutubeDL

FFMPEG_LOCATION = r"C:\ffmpeg\bin"

def get_ffmpeg_path() -> str:
    """Sistemde FFmpeg varlığını kontrol eder, varsa yolunu döner."""
    if shutil.which("ffmpeg"):
        return None
    common_paths = [
        FFMPEG_LOCATION,
        r"C:\ffmpeg\bin",
        r"C:\Program Files\ffmpeg\bin",
        r"C:\Program Files (x86)\ffmpeg\bin",
    ]
    for path in common_paths:
        if os.path.exists(os.path.join(path, "ffmpeg.exe")):
            return path
    return None

def sanitize_filename(name: str) -> str:
    """Dosya isimlerindeki geçersiz karakterleri temizler."""
    return re.sub(r'[\\/*?:"<>|]', "", name)

## 🌐 Arayüz Donmasını Engellemek: Threading Mantığı

Tkinter gibi masaüstü kütüphaneleri **tek bir iş parçacığı (Main Thread)** üzerinde çalışır. Eğer bu iş parçacığında `requests.get` veya `ydl.download` gibi internet hızına bağlı bloklayıcı ağ istekleri yaparsanız arayüz tamamen donar ve işletim sistemi "Yanıt Vermiyor" uyarısı gösterir.

Bu sorunu aşmak için:
1. **Thread:** Ağ isteklerini `threading.Thread(target=run, daemon=True).start()` ile arka plana atıyoruz.
2. **root.after:** Tkinter arayüz elemanlarını arka plan iş parçacığından doğrudan güncellemek güvenli değildir. Bunun yerine güncellemeleri kuyruğa alıp ana thread'e iletmek için `root.after(0, callback)` kullanıyoruz.

## ⚡ Video Bilgilerini Sorgulama (`fetch_info`)

`yt-dlp` motoru, indirme yapmadan da bir linkin metaverilerini çekebilir (`skip_download=True`). Bu sayede videonun başlığını ve önizleme (thumbnail) adresini alıp arayüze dinamik olarak yükleriz.

## 📥 İndirme ve Progress Hook Mantığı

`yt-dlp` indirme yaparken her paket indiğinde tetiklenen `progress_hooks` adında bir kanca (hook) sunar. Bu kanca bize indirilen yüzde oranını, hızı ve tahmini kalan süreyi verir. Biz de bu bilgileri alıp anlık olarak `Progressbar` ve durum etiketine yazdırırız.

## 🚀 Uygulamanın Tam Kodu ve Çalıştırılması

Aşağıdaki hücrede tüm arayüz sınıfını ve ana döngüyü çalıştırabilirsiniz. Uygulama bağımsız bir pencerede açılacaktır.

In [ ]:
class YouTubeDownloaderApp:
    def __init__(self, root):
        self.root = root
        self.root.title("YouTube Video & Ses İndirici")
        self.root.geometry("680x600")
        self.root.configure(bg="#f2f2f7")
        self.info_dict = None
        self.thumbnail_img = None
        self.create_ui()

    def create_ui(self):
        style = ttk.Style()
        style.theme_use('clam')
        style.configure("TFrame", background="#f2f2f7")
        style.configure("TLabel", background="#f2f2f7", foreground="#1c1c1e", font=("Helvetica", 11))
        style.configure("Header.TLabel", font=("Helvetica", 16, "bold"成品))
        style.configure("TButton", font=("Helvetica", 11, "bold"), foreground="#ffffff",
                        background="#007aff", padding=6)
        style.map("TButton", background=[("active", "#005bb5")])
        style.configure("TProgressbar", troughcolor="#d1d1d6", background="#007aff", thickness=8)

        main_frame = ttk.Frame(self.root, padding=20)
        main_frame.pack(expand=True, fill="both")

        ttk.Label(main_frame, text="YouTube Video & Ses İndirme Botu", style="Header.TLabel").pack(pady=(0,20))

        url_frame = ttk.Frame(main_frame)
        url_frame.pack(fill="x", pady=(0,10))
        ttk.Label(url_frame, text="YouTube Linki:", style="TLabel").pack(side="left")
        self.url_var = tk.StringVar()
        self.url_entry = ttk.Entry(url_frame, textvariable=self.url_var, width=45)
        self.url_entry.pack(side="left", padx=10)
        self.fetch_button = ttk.Button(url_frame, text="Bilgileri Getir", command=self.fetch_info)
        self.fetch_button.pack(side="left")

        info_frame = ttk.Frame(main_frame)
        info_frame.pack(fill="x", pady=(10,10))
        self.title_label = ttk.Label(info_frame, text="Başlık: Bekleniyor...", font=("Helvetica", 12, "bold"), wraplength=600)
        self.title_label.pack(anchor="w", pady=(0, 10))
        self.thumbnail_label = ttk.Label(info_frame, text="Önizleme resmi için yukarıdaki butona tıklayın.", anchor="center")
        self.thumbnail_label.pack(pady=10)

        option_frame = ttk.Frame(main_frame)
        option_frame.pack(fill="x", pady=(10,10))
        self.download_option = tk.StringVar(value="video")
        ttk.Label(option_frame, text="İndirilecek Tür:", style="TLabel").pack(side="left")
        
        self.video_radio = ttk.Radiobutton(option_frame, text="Video (MP4)", variable=self.download_option, value="video", command=self.toggle_resolution_combobox)
        self.video_radio.pack(side="left", padx=10)
        self.audio_radio = ttk.Radiobutton(option_frame, text="Ses (MP3)", variable=self.download_option, value="audio", command=self.toggle_resolution_combobox)
        self.audio_radio.pack(side="left", padx=10)
        
        self.res_label = ttk.Label(option_frame, text="Çözünürlük:")
        self.res_label.pack(side="left", padx=(20,0))
        self.resolution_var = tk.StringVar(value="En İyi")
        self.res_combobox = ttk.Combobox(option_frame, textvariable=self.resolution_var, values=["En İyi", "1080p", "720p", "480p"], state="readonly", width=8)
        self.res_combobox.pack(side="left", padx=10)

        button_frame = ttk.Frame(main_frame)
        button_frame.pack(pady=(15,10))
        self.download_button = ttk.Button(button_frame, text="İndirmeyi Başlat", command=self.download_media, state="disabled")
        self.download_button.pack(side="left", padx=10)
        self.open_folder_button = ttk.Button(button_frame, text="İndirilenleri Aç", command=self.open_download_folder)
        self.open_folder_button.pack(side="left")

        self.status_label = ttk.Label(main_frame, text="Kullanıma hazır.", font=("Helvetica", 10, "italic"))
        self.status_label.pack(pady=(10,5))
        self.progress_var = tk.DoubleVar()
        self.progress_bar = ttk.Progressbar(main_frame, orient="horizontal", length=500, mode="determinate", variable=self.progress_var)
        self.progress_bar.pack()

    def toggle_resolution_combobox(self):
        if self.download_option.get() == "audio":
            self.res_combobox.config(state="disabled")
        else:
            self.res_combobox.config(state="readonly")

    def fetch_info(self):
        url = self.url_var.get().strip()
        if not url:
            messagebox.showerror("Hata", "Lütfen geçerli bir YouTube linki girin.")
            return
        self.status_label.config(text="Video bilgileri alınıyor, lütfen bekleyin...")
        self.fetch_button.config(state="disabled")
        self.download_button.config(state="disabled")
        
        def run():
            try:
                opts = {'quiet': True, 'skip_download': True}
                ffmpeg_path = get_ffmpeg_path()
                if ffmpeg_path:
                    opts['ffmpeg_location'] = ffmpeg_path
                with YoutubeDL(opts) as ydl:
                    info = ydl.extract_info(url, download=False)
                self.root.after(0, lambda: self.on_fetch_success(info))
            except Exception as e:
                self.root.after(0, lambda: self.on_fetch_error(e))
        threading.Thread(target=run, daemon=True).start()

    def on_fetch_success(self, info):
        self.info_dict = info
        title = info.get('title', 'Bilinmiyor')
        self.title_label.config(text=f"Başlık: {title}")
        self.fetch_button.config(state="normal")
        self.download_button.config(state="normal")
        self.status_label.config(text="Bilgiler başarıyla alındı.")

        thumb_url = info.get('thumbnail')
        if thumb_url:
            self.thumbnail_label.config(text="Resim yükleniyor...")
            def load_thumbnail():
                try:
                    response = requests.get(thumb_url, timeout=10)
                    img = Image.open(io.BytesIO(response.content)).resize((320, 180), Image.Resampling.LANCZOS)
                    photo = ImageTk.PhotoImage(img)
                    self.root.after(0, lambda: self.update_thumbnail(photo))
                except:
                    self.root.after(0, lambda: self.thumbnail_label.config(text="Önizleme resmi yüklenemedi."))
            threading.Thread(target=load_thumbnail, daemon=True).start()
        else:
            self.thumbnail_label.config(text="Önizleme resmi yok.")

    def update_thumbnail(self, photo):
        self.thumbnail_img = photo
        self.thumbnail_label.config(image=self.thumbnail_img, text="")

    def on_fetch_error(self, err):
        self.fetch_button.config(state="normal")
        self.status_label.config(text="Bilgiler alınamadı.")
        messagebox.showerror("Hata", f"Video bilgileri alınırken hata oluştu:\n{err}")

    def download_media(self):
        if not self.info_dict:
            messagebox.showerror("Hata", "Önce 'Bilgileri Getir' butonuna basmalısınız.")
            return
        title = sanitize_filename(self.info_dict.get('title', 'video'))
        dtype = self.download_option.get()
        resolution = self.resolution_var.get()
        
        ffmpeg_path = get_ffmpeg_path()
        has_ffmpeg = (ffmpeg_path is not None) or (shutil.which("ffmpeg") is not None)

        if not has_ffmpeg:
            if dtype == 'audio' or resolution != 'En İyi':
                confirm = messagebox.askyesno(
                    "FFmpeg Bulunamadı",
                    "Sisteminizde FFmpeg kurulu bulunamadı.\n\n"
                    "FFmpeg olmadan yüksek çözünürlüklü video birleştirme ve MP3 dönüştürme işlemleri başarısız olabilir.\n"
                    "Yine de indirmeye devam etmek istiyor musunuz? (Olası en iyi kalitede tek parça dosya indirilecektir.)"
                )
                if not confirm:
                    return

        if dtype == 'video':
            if resolution == 'En İyi':
                if not has_ffmpeg:
                    fmt = 'best[ext=mp4]/best'
                else:
                    fmt = 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best'
            else:
                height = resolution.replace('p', '')
                if not has_ffmpeg:
                    fmt = f'best[height<={height}][ext=mp4]/best'
                else:
                    fmt = f'bestvideo[height<={height}][ext=mp4]+bestaudio[ext=m4a]/best'
            out = f"{title}.mp4"
            post = []
        else:
            if not has_ffmpeg:
                fmt = 'bestaudio'
                out = f"{title}.m4a"
                post = []
                messagebox.showwarning("Bilgi", "FFmpeg kurulu olmadığından dönüştürme yapılamadı. Dosya orijinal ses formatında (.m4a) indirilecektir.")
            else:
                fmt = 'bestaudio[ext=m4a]/bestaudio'
                out = f"{title}.mp3"
                post = [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3', 'preferredquality': '192'}]

        opts = {
            'format': fmt,
            'outtmpl': out,
            'progress_hooks': [self.progress_hook],
            'postprocessors': post,
            'quiet': True,
        }
        if ffmpeg_path:
            opts['ffmpeg_location'] = ffmpeg_path

        self.status_label.config(text="İndirme işlemi başlatılıyor...")
        self.progress_var.set(0)
        self.download_button.config(state="disabled")
        self.fetch_button.config(state="disabled")

        def run():
            try:
                with YoutubeDL(opts) as ydl:
                    ydl.download([self.info_dict['webpage_url']])
                self.root.after(0, lambda: self.on_download_success(out))
            except Exception as e:
                self.root.after(0, lambda: self.on_download_error(e))
        threading.Thread(target=run, daemon=True).start()

    def on_download_success(self, out_file):
        self.status_label.config(text=f"İndirme tamamlandı: {out_file}")
        self.download_button.config(state="normal")
        self.fetch_button.config(state="normal")
        messagebox.showinfo("Başarılı", f"Dosya başarıyla indirildi:\n{out_file}")

    def on_download_error(self, err):
        self.status_label.config(text="İndirme başarısız oldu.")
        self.download_button.config(state="normal")
        self.fetch_button.config(state="normal")
        messagebox.showerror("Hata", f"İndirme sırasında bir hata oluştu:\n{err}")

    def progress_hook(self, d):
        if d['status'] == 'downloading':
            try:
                p = float(d.get('_percent_str', '').replace('%', '').strip())
            except:
                p = 0
            speed = d.get('_speed_str', 'Bilinmiyor')
            eta = d.get('eta', 0)
            self.root.after(0, lambda: self.update_progress(p, speed, eta))
        elif d['status'] == 'finished':
            self.root.after(0, self.on_download_finished_hook)

    def update_progress(self, percent, speed, eta):
        self.progress_var.set(percent)
        self.status_label.config(text=f"İndiriliyor: %{percent:.1f} | Hız: {speed} | Kalan Süre: {eta} sn")

    def on_download_finished_hook(self):
        self.progress_var.set(100)
        self.status_label.config(text="İndirme tamamlandı, dosya birleştiriliyor/işleniyor...")

    def open_download_folder(self):
        try:
            os.startfile(os.getcwd())
        except:
            import subprocess
            subprocess.Popen(['xdg-open', os.getcwd()])

root = tk.Tk()
app = YouTubeDownloaderApp(root)
root.mainloop()